In [ ]:
#| hide
from pathlib import Path
from shutil import rmtree

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb

from nbskill.convert import py2nb
from nbskill.execute import exec_nb
from nbskill.foundation import cell_hash
from nbskill.mcp import create_mcp
from nbskill.read import read_nb, show_doc
from nbskill.review import diff_nb
from nbskill.write import update_cell, write_nb


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    return start


def _reset_demo_workspace(root):
    workspace = root / "nbs" / "data" / "readme-demo"
    if workspace.exists():
        rmtree(workspace)
    workspace.mkdir(parents=True)
    return workspace


project_root = _find_project_root()

# nbskill

`nbskill` is a small toolkit for working with nbdev notebooks as source code. It gives agents and humans notebook-aware commands for reading, writing, executing, reviewing, converting, and serving notebooks through MCP.

The central idea is simple: keep the notebook as the source of truth, but give automation stable handles so it can make small, reviewable changes without touching raw JSON.

## The problem this project solves

Raw notebooks are awkward for coding agents. Cell ids, outputs, metadata, Markdown, and code are mixed together in JSON, while nbdev expects the meaningful implementation to stay in notebooks and export clean Python modules from there.

That mismatch matters in practice. A small source edit can accidentally preserve stale outputs, overwrite the wrong cell after another edit, or hide the useful code change inside a noisy JSON diff. Agents also need to understand whether a cell is documentation, setup, exported code, a test, or a show-off example before they touch it.

`nbskill` turns that into a safer workflow:

- `read_nb` and `show_doc` expose compact notebook views.
- `write_nb` and `update_cell` edit cells by ids, chapters, text, and source hashes.
- `exec_nb` runs notebooks with local project imports available and records visible outputs.
- `diff_nb` and `style_check` make review focused on code cells and style hints.
- `py2nb` and `py2nbs` bootstrap notebooks from Python modules when a project is moving toward nbdev.
- `nbskill_mcp` exposes the same operations to Codex, Claude, or any MCP client.

## How the notebooks fit together

The notebooks in `nbs/` are ordered like the toolchain itself:

1. `00_foundation.ipynb` defines the private parsing, cell, chapter, and CLI helpers used everywhere else.
2. `01_read.ipynb` makes notebooks readable without JSON noise.
3. `02_write.ipynb` applies safe cell edits and exports when requested.
4. `03_execute.ipynb` executes notebooks in the local project context.
5. `04_review.ipynb` keeps review centered on code-cell diffs and style feedback.
6. `05_convert.ipynb` turns Python files into nbdev notebooks.
7. `06_skill.ipynb` installs the bundled agent skill.
8. `07_mcp.ipynb` wraps the functions as an MCP server.
9. `08_edit_interactive.ipynb` runs a bounded edit loop over one notebook.
10. `09_parallel.ipynb` provides locks so concurrent notebook operations stay orderly.
11. `10_graph.ipynb` builds a static symbol graph for definitions, callers, and private-helper reports.

## A tiny notebook to work on

The examples below create a temporary notebook, then use the same public functions that are exposed as CLI commands and MCP tools. Nothing here edits this repository.

In [ ]:
workspace = _reset_demo_workspace(project_root)
demo_nb = workspace / "demo.ipynb"

_write_nb(new_nb([
    mk_cell("## A tiny notebook", cell_type="markdown"),
    mk_cell("x = 2\nx + 3", cell_type="code"),
]), demo_nb)

print(demo_nb)

## Reading: start with a compact map

`read_nb` is the first tool to reach for. It shows cell ids, semantic cell classes, and short source previews so an editor can decide where to operate before asking for full source.

This is needed because raw notebook JSON answers the wrong question first: it shows serialization details before intent. The compact map lets an agent find the cell that matters, notice nearby tests or docs, and copy the exact id or hash needed for a later guarded edit.

In [ ]:
_ = read_nb(str(demo_nb), context="overview", show_ids=True)

## Writing: add cells without raw notebook JSON

`write_nb` accepts cell blocks separated by `---`. The `%%markdown` and `%%code` markers make each new cell explicit, while `export=False` keeps this temporary example from running nbdev export.

This is useful when adding examples, tests, or explanatory sections. The caller describes notebook cells as cells, not JSON objects, so nbskill can preserve notebook structure and clear stale execution state where needed.

In [ ]:
_ = write_nb(
    str(demo_nb),
    "%%markdown\n## Result\nThe next cell computes from the earlier value.\n---\n%%code\nanswer = x * 10\nanswer",
    export=False,
)
_ = read_nb(str(demo_nb), context="overview", show_ids=True)

## Updating: use ids and hashes for precise edits

A notebook cell id tells `update_cell` which cell to change. A source hash is an optional guard: if another edit changed the cell first, the update fails instead of overwriting stale content.

That guard is the main safety feature for collaborative notebook editing. An agent can read a cell, propose a narrow replacement, and prove it is still editing the same source it inspected rather than a newer version from another human or tool.

In [ ]:
answer_cell = next(cell for cell in _read_nb(demo_nb).cells if "answer = x * 10" in cell.source)
hash_before = cell_hash(answer_cell, n=None)

_ = update_cell(
    str(demo_nb),
    "answer = x * 12\nanswer",
    cell_id=answer_cell.id,
    source_hash=hash_before,
    export=False,
)
_ = read_nb(str(demo_nb), cell_id=answer_cell.id, context="precise", show_ids=True)

## Executing: run the notebook as a notebook

`exec_nb` uses `execnb` and adds the notebook directory plus the project root to the import path. That lets tests and examples behave like they do inside an nbdev project.

This matters because many notebook bugs only appear when cells are run in order with the same imports, fixtures, and local package path a real user gets. A normal Python import check can miss that story; executing the notebook checks the literate source itself.

In [ ]:
_ = exec_nb(str(demo_nb), timeout=5, show_output=True)

## Reviewing: look at behavior and code changes

`show_doc` answers the question "why is this symbol here?" by combining nearby Markdown, the symbol signature, and optional source. `diff_nb` keeps review focused on code-cell source rather than outputs and metadata.

These tools keep review at the level a maintainer cares about. `show_doc` reconstructs the local rationale around a function, while `diff_nb` filters out notebook churn so a reviewer can see whether the implementation changed.

In [ ]:
_ = show_doc(str(project_root / "nbs/02_write.ipynb"), "write_nb", context=1, source=False)
_ = diff_nb(str(project_root / "nbs/02_write.ipynb"), ref_a=None)

## Converting: bootstrap nbdev notebooks from Python

`py2nb` parses Python with `ast`, creates one nbdev notebook, and keeps exports explicit. It is useful when a project starts in `.py` files but wants to move toward literate notebooks.

In [ ]:
sample_py = workspace / "sample_tool.py"
sample_py.write_text("def double(x):\n    return x * 2\n", encoding="utf-8")
converted_nb = workspace / "sample_tool.ipynb"

_ = py2nb(str(sample_py), dest=str(converted_nb))
_ = read_nb(str(converted_nb), context="overview", show_ids=True)

## Serving the workflow through MCP

The MCP server in `07_mcp.ipynb` registers the same operations as tools. The server layer is intentionally thin: it captures stdout, holds notebook locks around file operations, and delegates the real behavior back to the notebook-defined functions.

In [ ]:
mcp = create_mcp()
type(mcp).__name__

## A realistic agent workflow

A typical agent session should be small and reversible. First check that the MCP server is connected, then read the notebook at overview level before selecting one precise cell to edit.

```python
healthcheck()
read_nb(path="nbs/02_write.ipynb", context="overview", show_ids=True)
read_nb(path="nbs/02_write.ipynb", cell_id="abc123", context="precise", show_ids=True)
```

After inspecting the precise cell, carry its `source_hash` into the edit. That turns the update into a guarded write: if the cell changed after the read, nbskill refuses the stale edit instead of guessing.

```python
update_cell(
    path="nbs/02_write.ipynb",
    cell_id="abc123",
    source_hash="7f3a91c0d422",
    new="def target():\n    return 'updated'",
    export=False,
)
```

Finish with a review or execution tool depending on what changed. Use `diff_nb` for implementation edits and `exec_nb` when the notebook behavior needs to be checked end to end.

```python
diff_nb(path="nbs/02_write.ipynb")
exec_nb(path="nbs/02_write.ipynb", timeout=10, show_output=True)
```

<!-- nbskill-skill:start -->
# Jupyter Notebooks

Use this skill when a repository treats notebooks as source files, especially nbdev projects where `nbs/*.ipynb` exports to Python modules. Keep the active workflow small: inspect notebooks, edit notebooks, execute notebooks, and use references only when the task needs a supporting tool.

## Setup

Install the local package and register the MCP server:

```bash
uv tool install --editable . --force
codex mcp add nbskill -- nbskill_mcp
claude mcp add nbskill -- nbskill_mcp
```

Prefer the MCP server when it is available. Call `healthcheck` first to confirm the server is alive, see the installed version, inspect capabilities, and confirm concurrency policy. If MCP tools are missing, run `uv run nbskill_status` to see the canonical command names and reconnect instructions, then restart or reconnect the MCP client after reinstalling nbskill.

## Core Workflow

1. Use `read_nb` to inspect notebooks without raw JSON. Start with `context="overview"` and use `context="precise"` when you need numbered source for a selected cell. Quote query values with spaces, such as `query='contains="def write_nb"'`.
2. Use `write_nb` to add notebook cells by cell id, chapter, or full-notebook replacement. Use `cells_file` for multiline additions.
3. Use `update_cell` for precise edits to an existing cell by id, text replacement, or line range. Use `new_file` for multiline replacements and `source_hash` when stale context should fail instead of overwriting newer work.
4. Use `batch_edit_nb` for coordinated multi-cell or multi-notebook edits. Start with `dry_run=True`, include source hashes for guarded cells, and inspect the printed diffs before writing.
5. Use `style_check` as the main hygiene report for large cells, mixed semantic cells, duplicate imports, cell-order problems, and global tool usage/problems.
6. Use `diff_nb` to inspect code-cell diffs; use `git diff` for Markdown or documentation changes. nbskill metadata-only changes are summarized instead of expanded.
7. Use `exec_nb` to run a notebook, chapter, or cells up to an id, then inspect visible outputs and errors.

Stay notebook-first: edit `nbs/*.ipynb` source notebooks, not generated `.py` files. Use stable cell ids and source hashes from `read_nb --show_ids` when an edit must be guarded against stale context. Functions beginning with `_` are notebook-local unless deliberately promoted to a public helper.

## CLI Fallback

Use CLI commands only when MCP tools are unavailable or final verification must run in the project environment:

```bash
uv run nbskill_status
uv run read_nb nbs/02_write.ipynb --context overview --show_ids
uv run read_nb nbs/02_write.ipynb --context precise --query 'contains="def write_nb"'
uv run write_nb nbs/02_write.ipynb --after_id abc123 --cells_file /tmp/cells.txt --no-export
uv run write_nb nbs --old_str old_name --new_str new_name --dry_run --show_cells --no-export
uv run update_cell nbs/02_write.ipynb --cell_id abc123 --source_hash 7f3a91c0d422 --new_file /tmp/cell.txt --no-export
uv run batch_edit_nb --plan_file /tmp/nbskill-plan.json --dry_run
uv run diff_nb nbs/02_write.ipynb
uv run style_check nbs --delete-after-output
uv run private_symbol_report --path nbs
uv run exec_nb nbs/03_execute.ipynb --up2id abc123 --timeout 10
```

Cell blocks for `write_nb` are separated by a line containing only `---`; start blocks with `%%markdown`, `%%md`, `%%code`, or `%%raw` when the cell type matters.

## References

Open references only when the core workflow is not enough:

- `references/mcp-tools.md` for detailed MCP behavior, reconnect notes, and concurrency behavior.
- `references/cli-fallbacks.md` for shell-friendly command patterns.
- `references/conversion.md` for converting Python files or folders with `py2nb`.
- `references/extended-tools.md` for symbol docs, review, graph reports, and edit-interactive plans.
<!-- nbskill-skill:end -->

## Sugar features and notebook policy

`read_nb` prints numbered source when a query resolves to one cell, even in overview mode. Semantic cell information is persisted in `metadata["nbskill"]`, including the notebook cell type, semantic types such as `exported_code` or `test_cell`, and the source hash that keeps the metadata honest. Quote query values that contain spaces, for example `query='contains="def target"'`; semicolons separate multiple selections.

`write_nb` can also do literal replacements across a notebook file, directory, or glob by passing `old_str` and `new_str`. This is intended for exact-match renames across notebooks; use `dry_run=True, show_cells=True` before writing to see touched notebook paths, cell ids, match counts, and compact diffs.

`diff_nb` stays focused on code-cell changes. Use `git diff` for Markdown and documentation edits; `metadata["nbskill"]` changes are hidden from the diff body and summarized in one line when they are the only notebook changes.

`symbol_graph(path="nbs", symbol="name")` reports definitions, callers, and callees with notebook paths and cell ids. `private_symbol_report(path="nbs")` surfaces imported cross-notebook calls to private helpers, including `from nbskill.module import _helper` policy violations.

Functions, classes, and methods starting with `_` are excluded from nbdev's exported `__all__`. Treat them as notebook-local implementation details. If another notebook needs a helper, promote it to a public name first; do not casually import private helpers across notebooks.

<!-- nbskill-skill:start -->
# Jupyter Notebooks

Use this skill when a repository treats notebooks as source files, especially nbdev projects where `nbs/*.ipynb` exports to Python modules. Keep the active workflow small: inspect notebooks, edit notebooks, execute notebooks, and use references only when the task needs a supporting tool.

## Setup

Install the local package and register the MCP server:

```bash
uv tool install --editable . --force
codex mcp add nbskill -- nbskill_mcp
claude mcp add nbskill -- nbskill_mcp
```

Prefer the MCP server when it is available. Call `healthcheck` first to confirm the server is alive, see the installed version, inspect capabilities, and confirm concurrency policy. After reinstalling or exporting new MCP tool signatures, fully restart or reconnect the MCP client so it refreshes cached schemas.

## Core Workflow

1. Use `read_nb` to inspect notebooks without raw JSON. Start with `context="overview"` and use `context="precise"` when you need numbered source for a selected cell. Quote query values with spaces, such as `query='contains="def write_nb"'`.
2. Use `write_nb` to add notebook cells by cell id, chapter, or full-notebook replacement. Use `cells_file` for multiline additions.
3. Use `write_nb(path, old_str="old", new_str="new", dry_run=True, show_cells=True)` to preview exact literal replacements with touched cell ids and compact diffs before writing.
4. Use `update_cell` for precise edits to an existing cell by id, text replacement, or line range. Use `source_hash` when stale context should fail instead of overwriting newer work.
5. Use `diff_nb` to inspect code-cell diffs; use `git diff` for Markdown or documentation changes. nbskill metadata-only changes are summarized instead of expanded.
6. Use `exec_nb` to run a notebook, chapter, or cells up to an id, then inspect visible outputs and errors.

Stay notebook-first: edit `nbs/*.ipynb` source notebooks, not generated `.py` files. Use stable cell ids and source hashes from `read_nb --show_ids` when an edit must be guarded against stale context. Functions beginning with `_` are notebook-local unless deliberately promoted to a public helper.

## CLI Fallback

Use CLI commands only when MCP tools are unavailable or final verification must run in the project environment:

```bash
uv run read_nb nbs/02_write.ipynb --context overview --show_ids
uv run read_nb nbs/02_write.ipynb --context precise --query 'contains="def write_nb"'
uv run write_nb nbs/02_write.ipynb --after_id abc123 --cells_file nbs/data/cells.txt --no-export
uv run write_nb nbs --old_str old_name --new_str new_name --dry_run --show_cells --no-export
uv run update_cell nbs/02_write.ipynb "replacement line" --cell_id abc123 --line_range 3 --source_hash 7f3a91c0d422 --no-export
uv run diff_nb nbs/02_write.ipynb
uv run private_symbol_report --path nbs
uv run exec_nb nbs/03_execute.ipynb --up2id abc123 --timeout 10
```

Cell blocks for `write_nb` are separated by a line containing only `---`; start blocks with `%%markdown`, `%%md`, `%%code`, or `%%raw` when the cell type matters.

## References

Open references only when the core workflow is not enough:

- `references/mcp-tools.md` for detailed MCP behavior, reconnect notes, and concurrency behavior.
- `references/cli-fallbacks.md` for shell-friendly command patterns.
- `references/conversion.md` for converting Python files or folders with `py2nb`.
- `references/extended-tools.md` for symbol docs, review, graph reports, and edit-interactive plans.
<!-- nbskill-skill:end -->

## Agent editing policy
 
We want notebook edits to stay small, readable, and reviewable. A good cell does one semantic job: imports, exported code, private implementation, tests, or a visible example. `style_check` reports cells that mix those jobs, code cells with more than two top-level functions or more than twenty non-directive lines, and test cells that check multiple problems at once.
 
Do not hide broad changes inside oversized cells, duplicate imports across the same notebook scope, or bundle several unrelated assertions into one test cell. Prefer one-problem-at-a-time tests in separate cells, keep examples after the behavior they demonstrate, and use `cells_file`, `new_file`, or stdin for complex multiline CLI edits.